# 영화 랭킹 정보 조회 및 포스터 이미지 저장하기

In [2]:
import requests
from bs4 import BeautifulSoup
from io import BytesIO
from PIL import Image
import re
import os

movie_ranking = requests.get("https://www.moviechart.co.kr/rank/realtime/index/image")

image_dir = 'images'
if not os.path.exists(image_dir):
  os.makedirs(image_dir)

pattern = r'[\\/:"*?<>|]' #파일이나 폴더 이름으로 절대 사용할 수 없는 특수문자 9가지

if movie_ranking.status_code == 200:
  print("영화 정보를 출력합니다.")
  soup = BeautifulSoup(movie_ranking.content, 'html.parser')
  
  #content > div.wArea.space > div.movieBox > ul > li:nth-child(2) > div > div.movie-title > h3 > a
  movie_title_list = soup.select(".movieBox-list .movie-title a") # 영화 이름 <a> 요소 목록
  movie_image_list = soup.select(".movieBox-list .movieBox-item img") #img 태그 요소
  print(f"수집한 영화 수: {len(movie_title_list)}")

  for movie_title, movie_image in zip(movie_title_list, movie_image_list):
    print(movie_title.text, movie_image.get('src'))  #이름 요소의 텍스트, 이미지 요소의 src속성의 값
    
    image_src = movie_image.get('src')
    image_response = requests.get("https://www.moviechart.co.kr" + image_src)
    img = Image.open(BytesIO(image_response.content))
    image_filename = re.sub(pattern, '', movie_title.text) #파일이름 지정, 정규식
    img.save(os.path.join(image_dir, image_filename + ".png"))
    print(movie_title.text, )  
else:
  print("페이지에 연결할 수 없습니다.")

영화 정보를 출력합니다.
수집한 영화 수: 20
왕과 사는 남자 /thumb?width=178&height=267&m_code=20242837&source=https://admin.moviechart.co.kr/assets/upload/movie/260108003526_8322.jpg
왕과 사는 남자
휴민트 /thumb?width=178&height=267&m_code=20241266&source=https://admin.moviechart.co.kr/assets/upload/movie/260112024651_6301.jpg
휴민트
초속 5센티미터 /thumb?width=178&height=267&m_code=20259583&source=https://admin.moviechart.co.kr/assets/upload/movie/260213052538_7190.jpg
초속 5센티미터
너자 2 /thumb?width=178&height=267&m_code=20261181&source=https://admin.moviechart.co.kr/assets/upload/movie/260202063756_4036.jpg
너자 2
슬라이드 스트럼 뮤트 /thumb?width=178&height=267&m_code=20261450&source=https://admin.moviechart.co.kr/assets/upload/movie/260219042530_2928.jpg
슬라이드 스트럼 뮤트
넘버원 /thumb?width=178&height=267&m_code=20252373&source=https://admin.moviechart.co.kr/assets/upload/movie/260123065552_5690.jpg
넘버원
햄넷 /thumb?width=178&height=267&m_code=20250365&source=https://admin.moviechart.co.kr/assets/upload/movie/260202063420_3704.jpg
햄넷
부흥 /thumb?wid

In [6]:
import requests
from bs4 import BeautifulSoup
from io import BytesIO
from PIL import Image
import re
import os

URL = "https://www.moviechart.co.kr/rank/realtime/index/image"
BASE = "https://www.moviechart.co.kr"

image_dir = "images"
os.makedirs(image_dir, exist_ok=True)

pattern = r'[\\/:"*?<>|]'

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Referer": BASE + "/",
    "Accept-Language": "ko-KR,ko;q=0.9",
})

movie_ranking = session.get(URL, timeout=20)

if movie_ranking.status_code != 200:
    print("페이지에 연결할 수 없습니다.", movie_ranking.status_code)
    raise SystemExit

print("영화 정보를 출력합니다.")
soup = BeautifulSoup(movie_ranking.content, "html.parser")

title_list = soup.select(".movieBox-list .movie-title a")
img_list = soup.select(".movieBox-list .movieBox-item img")
print(f"수집한 영화 수: {len(title_list)}")

for rank, (movie_title, movie_image) in enumerate(zip(title_list, img_list), start=1):
    title = movie_title.get_text(strip=True)

    # lazy-load 대비: src 외 속성도 같이 확인
    image_src = (
        movie_image.get("data-src")
        or movie_image.get("data-original")
        or movie_image.get("data-lazy")
        or movie_image.get("src")
        or ""
    )

    if not image_src:
        print(f"[{rank:02d}] src 없음: {title}")
        continue

    image_url = image_src if image_src.startswith("http") else BASE + image_src
    print(f"[{rank:02d}] {title} | {image_url}")

    # 이미지 다운로드
    image_response = session.get(image_url, timeout=20, allow_redirects=True)

    # ✅ Content-Type으로 판단하지 말고, PIL로 실제 이미지인지 판별
    try:
        img = Image.open(BytesIO(image_response.content))
        img.load()  # 실제 로드 강제(깨진 데이터면 여기서 터짐)
    except Exception as e:
        # 이미지가 아니면 보통 HTML(차단/에러)일 확률이 큼
        ct = image_response.headers.get("Content-Type", "")
        print(f"   저장 실패(이미지 파싱 불가): status={image_response.status_code}, ct={ct}, err={e}")
        print("   응답 앞부분:", image_response.text[:120].replace("\n", " "))
        continue

    # 파일명 정리 + 저장
    image_filename = re.sub(pattern, "", title).strip() or f"movie_{rank:02d}"
    save_path = os.path.join(image_dir, f"{rank:02d}_{image_filename}.png")
    img.save(save_path)
    print("   저장 완료:", save_path)

영화 정보를 출력합니다.
수집한 영화 수: 20
[01] 왕과 사는 남자 | https://www.moviechart.co.kr/thumb?width=178&height=267&m_code=20242837&source=https://admin.moviechart.co.kr/assets/upload/movie/260108003526_8322.jpg
   저장 완료: images\01_왕과 사는 남자.png
[02] 휴민트 | https://www.moviechart.co.kr/thumb?width=178&height=267&m_code=20241266&source=https://admin.moviechart.co.kr/assets/upload/movie/260112024651_6301.jpg
   저장 완료: images\02_휴민트.png
[03] 초속 5센티미터 | https://www.moviechart.co.kr/thumb?width=178&height=267&m_code=20259583&source=https://admin.moviechart.co.kr/assets/upload/movie/260213052538_7190.jpg
   저장 완료: images\03_초속 5센티미터.png
[04] 너자 2 | https://www.moviechart.co.kr/thumb?width=178&height=267&m_code=20261181&source=https://admin.moviechart.co.kr/assets/upload/movie/260202063756_4036.jpg
   저장 완료: images\04_너자 2.png
[05] 슬라이드 스트럼 뮤트 | https://www.moviechart.co.kr/thumb?width=178&height=267&m_code=20261450&source=https://admin.moviechart.co.kr/assets/upload/movie/260219042530_2928.jpg
   저장 완료: images

In [3]:
import requests
from bs4 import BeautifulSoup
from io import BytesIO
from PIL import Image
import re
import os
from urllib.parse import urlparse, parse_qs

movie_ranking = requests.get("https://www.moviechart.co.kr/rank/realtime/index/image")

image_dir = 'images2'
if not os.path.exists(image_dir):
  os.makedirs(image_dir)

pattern = r'[\\/:"*?<>|]' 

if movie_ranking.status_code == 200:
  print("영화 정보를 출력합니다.")
  soup = BeautifulSoup(movie_ranking.content, 'html.parser')
  movie_title_list = soup.select(".movieBox-list .movie-title a")
  movie_image_list = soup.select(".movieBox-list .movieBox-item img")
  print(f"수집한 영화 수: {len(movie_title_list)}")

  for movie_title, movie_image in zip(movie_title_list, movie_image_list):
    url = movie_image.get('src')
    parsed_url = urlparse(url)
    query_params = parse_qs(parsed_url.query)
    image_src = query_params.get('source', [None])[0]
    # image_response = requests.get('https://www.moviechart.co.kr' + image_src)
    image_response = requests.get(image_src)
    img = Image.open(BytesIO(image_response.content))
    image_filename = re.sub(pattern, '', movie_title.text)
    img.save(os.path.join(image_dir, image_filename + '.png'))
    print(movie_title.text, )
else:
  print("페이지에 연결할 수 없습니다.")

영화 정보를 출력합니다.
수집한 영화 수: 20
왕과 사는 남자
휴민트
초속 5센티미터
너자 2
슬라이드 스트럼 뮤트
넘버원
햄넷
부흥
매드 댄스 오피스
신의악단
28년 후: 뼈의 사원
렌탈 패밀리: 가족을 빌려드립니다
몬테크리스토 백작
호퍼스
만약에 우리
폭풍의 언덕
점보
아기 티라노 디보: 초식이지만 괜찮아!
직사각형, 삼각형
아바타: 불과 재
